# Exact-path evaluation: ordered nodes and relationships

**Question:** Does an expected ordered path, with the expected relationships, occur in the saved results, and at what rank?

**Current limitation:** The 32 drug/chemical–disease test assets define unordered expected nodes and a minimum node count. They do **not** define an ordered chain or the predicate/direction of every hop. Their `predicate_id: biolink:related_to` describes the query, not an expected predicate for each path hop. The order of `path_nodes` is not a validated mechanism.

Consequently, the default exact-evaluation status is **`missing_reference`**, not a failure and not a rank. We must supply a reference before measuring exact-path accuracy. The separate node-only check below evaluates the Imatinib → KIT → histamine → mast cell → asthma sequence discussed in the conversation, without asserting that it is a validated biological pathway.

This notebook uses the saved November 7, 2026 TEST results (500 returned paths per case). It sends no queries. Outputs are saved; rerun with `.local/TestHarness/.venv/bin/python`.

**Folder guide:** notebook outputs go to `../exports/exact_paths/`. Shared generated data is in `../data/`, and editable reference pathways are in `../references/`. See `../START_HERE.txt` for the reading and execution order.

In [1]:
from pathlib import Path
from collections import defaultdict
import json
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "test_results/pathfinder-test-full-20261107T192900Z").is_dir())
BASE = ROOT / "pathfiner_aragorn_exp"
DATA = BASE / "data"
REFERENCES = BASE / "references"
OUT = BASE / "exports/exact_paths"
OUT.mkdir(parents=True, exist_ok=True)
DATA.mkdir(parents=True, exist_ok=True)
RUN = ROOT / "test_results/pathfinder-test-full-20261107T192900Z"
paths = json.loads((DATA / "drug_disease_path_details.json").read_text())
references = json.loads((REFERENCES / "exact_path_references.json").read_text())
by_case = defaultdict(list)
for path in paths:
    by_case[path["case_id"]].append(path)
assert len(paths) == 3000 and len(by_case) == 6
normalization = {case: json.loads((RUN / case / "normalized_curies.json").read_text()) for case in by_case}

def canonical(case, identifier):
    # Use only normalization saved during this run. Unlisted IDs remain literal.
    return normalization[case].get(identifier, identifier)


## What the existing benchmark establishes

These counts reproduce the original asset check from the saved bound graph edges. The helper retains TestHarness's alternative-ID behavior. No order or predicate requirement is added here. The table reports how many expected identifiers were found together in a single path; it does not imply the listed nodes form one biological chain.

In [2]:
def harness_matched_nodes(graph_edge_ids, edges, groups):
    matched = set()
    for eid in graph_edge_ids:
        edge = edges[eid]
        for group in groups:
            if not any(curie in matched for curie in group):
                if edge["subject"] in group:
                    matched.add(edge["subject"])
                if edge["object"] in group:
                    matched.add(edge["object"])
    return matched

asset_records = []
for case_id, case_paths in by_case.items():
    response_files = {p["response_file"] for p in case_paths}
    assert len(response_files) == 1
    message = json.loads((ROOT / next(iter(response_files))).read_text())["message"]
    edges = message["knowledge_graph"]["edges"]
    for entry in json.loads((RUN / case_id / "assets.json").read_text()):
        asset = entry["asset"]
        groups = [{canonical(case_id, i) for i in n["ids"]} for n in asset["path_nodes"]]
        observations = []
        for path in case_paths:
            graph = message["auxiliary_graphs"][path["graph_id"]]
            matched = harness_matched_nodes(graph["edges"], edges, groups)
            observations.append((path["rank"], len(matched)))
        ranks = sorted(rank for rank, count in observations if count >= asset["minimum_required_path_nodes"])
        asset_records.append({
            "case_id": case_id, "asset_id": asset["id"], "query": case_paths[0]["query"],
            "expected_nodes_unordered": "; ".join(n["name"] for n in asset["path_nodes"]),
            "listed_node_groups": len(groups), "harness_minimum": asset["minimum_required_path_nodes"],
            "max_expected_ids_in_one_path": max(count for rank, count in observations),
            "harness_status": "PASSED" if ranks else "FAILED",
            "harness_first_rank": min(ranks) if ranks else None,
        })
assets = pd.DataFrame(asset_records)
assert len(assets) == 32 and (assets.harness_status == "PASSED").sum() == 10
saved = pd.read_csv(BASE / "exports/ranks/drug_disease_all_32_asset_results.csv")
check = assets.merge(saved, on=["case_id", "asset_id"], suffixes=("", "_saved"), validate="one_to_one")
assert (check.harness_status == check.status).all()
assert (check.harness_first_rank.fillna(-1) == check.first_matching_rank.fillna(-1)).all()
with pd.option_context("display.max_colwidth", None):
    display(assets.loc[assets.harness_minimum > 1,
        ["case_id", "query", "listed_node_groups", "harness_minimum", "max_expected_ids_in_one_path", "harness_status"]])


,case_id,query,listed_node_groups,harness_minimum,max_expected_ids_in_one_path,harness_status
3,TestCase_0,Imatinib → asthma,3,2,1,FAILED
6,TestCase_6,Imatinib → allergic asthma,3,2,1,FAILED
9,TestCase_2,Fluoxetine → long COVID-19,5,2,1,FAILED
16,TestCase_3,Acetylsalicylic acid → colorectal cancer,6,2,1,FAILED
24,TestCase_11,acetylsalicylate → colorectal cancer,6,2,0,FAILED
31,TestCase_16,Sildenafil → Alzheimer disease,3,2,1,FAILED


## The specific Imatinib node sequence discussed

This is a **node-only candidate check**, not an exact relationship benchmark. IDs use the saved asset alternatives and normalization. An exact sequence allows no extra nodes; an ordered subsequence allows extra intermediate nodes. Neither requires predicates here. Failure of even the unordered all-node check rules out this candidate among the returned paths regardless of predicates.

In [3]:
case_id = "TestCase_0"
combined = next(e["asset"] for e in json.loads((RUN / case_id / "assets.json").read_text())
                if e["asset"]["id"] == "Asset_1")
expected_by_name = {n["name"]: n["ids"] for n in combined["path_nodes"]}
candidate_groups = [[combined["source_input_id"]], expected_by_name["KIT"],
                    expected_by_name["Histamine"], expected_by_name["Mast Cell"],
                    [combined["target_input_id"]]]
candidate_groups = [{canonical(case_id, i) for i in group} for group in candidate_groups]

def ordered_nodes_match(sequence, groups, allow_extra_nodes=False):
    if not allow_extra_nodes:
        return len(sequence) == len(groups) and all(n in g for n, g in zip(sequence, groups))
    position = 0
    for node in sequence:
        if position < len(groups) and node in groups[position]:
            position += 1
    return position == len(groups)

candidate_rows = []
for label, predicate in [
    ("All candidate nodes present, any order", lambda seq: all(set(seq) & g for g in candidate_groups)),
    ("Candidate node order, extra nodes allowed", lambda seq: ordered_nodes_match(seq, candidate_groups, True)),
    ("Exact candidate node sequence, no extra nodes", lambda seq: ordered_nodes_match(seq, candidate_groups)),
]:
    hits = [p["rank"] for p in by_case[case_id] if predicate(p["node_ids"])]
    candidate_rows.append({"check": label, "returned_paths_checked": len(by_case[case_id]),
                           "matching_paths": len(hits), "first_rank": min(hits) if hits else None})
candidate_results = pd.DataFrame(candidate_rows)
display(candidate_results)
print("Candidate: Imatinib → KIT → histamine → mast cell → asthma")
print("Relationship check: undefined; no per-hop reference predicates/directions supplied.")


,check,returned_paths_checked,matching_paths,first_rank
0,"All candidate nodes present, any order",500,0,None
1,"Candidate node order, extra nodes allowed",500,0,None
2,"Exact candidate node sequence, no extra nodes",500,0,None


Candidate: Imatinib → KIT → histamine → mast cell → asthma
Relationship check: undefined; no per-hop reference predicates/directions supplied.


## Supply exact references

`../references/exact_path_references.json` contains one template per asset. The original unordered node clues are preserved separately. Fill in:

- `ordered_nodes`: a list of ID lists, including source and target. Each inner list gives allowed aliases for that one node position. This list explicitly defines order.
- `edge_requirements`: one object per hop, containing `predicates` (a nonempty list of full Biolink predicates) and `direction` (`forward`, `reverse`, or explicitly `either`, relative to traversal). Optional `qualifiers` is a list of required qualifier type/value objects.
- `reference_source`: the provenance for the chosen pathway, e.g. a user-specified hypothesis or a curated reference. The evaluator does not infer biological validity.

**Exact matching** requires the full node sequence with no additional nodes. Each hop needs an actual bound-graph edge satisfying its predicate, direction, and any requested qualifiers together on that same edge. Parallel edges are alternatives. Qualifiers not specified by the reference are unconstrained. Predicates are matched literally; no automatic hierarchy/inverse mapping is performed. Nested support graphs are not substituted for path edges.

A complete reference can return `found` or `not_found_in_returned_paths`, with the first rank and all matching ranks. Incomplete references remain `missing_reference` and are excluded from the success-rate denominator. `not_found_in_returned_paths` refers only to the saved 500 results, not the entire knowledge graph.

In [4]:
def validate_reference(reference):
    missing = []
    nodes = reference.get("ordered_nodes")
    hops = reference.get("edge_requirements")
    if not isinstance(nodes, list) or len(nodes) < 2 or any(
        not isinstance(group, list) or not group or any(not isinstance(i, str) or not i for i in group)
        for group in nodes
    ):
        missing.append("ordered_nodes")
    if not isinstance(hops, list) or not hops:
        missing.append("edge_requirements")
    elif any(not isinstance(h, dict) or not h.get("predicates") or
             not isinstance(h.get("predicates"), list) or
             any(not isinstance(p, str) or not p.startswith("biolink:") for p in h["predicates"]) or
             h.get("direction") not in ("forward", "reverse", "either") or
             not isinstance(h.get("qualifiers", []), list) or
             any(not isinstance(q, dict) or "qualifier_type_id" not in q or "qualifier_value" not in q
                 for q in h.get("qualifiers", [])) for h in hops):
        missing.append("per-hop predicates/directions/qualifiers")
    if isinstance(nodes, list) and isinstance(hops, list) and len(hops) != len(nodes) - 1:
        missing.append("one edge requirement per hop")
    if not reference.get("reference_source"):
        missing.append("reference_source")
    return missing


def edge_matches(edge, source, target, requirement):
    forward = (edge["subject"], edge["object"]) == (source, target)
    reverse = (edge["subject"], edge["object"]) == (target, source)
    direction = requirement["direction"]
    direction_ok = (direction == "forward" and forward) or (direction == "reverse" and reverse) or (
        direction == "either" and (forward or reverse))
    return (direction_ok and edge["predicate"] in requirement["predicates"] and
            all(q in edge.get("qualifiers", []) for q in requirement.get("qualifiers", [])))


def exact_path_matches(path, reference, edges):
    groups = [{canonical(reference["case_id"], i) for i in group} for group in reference["ordered_nodes"]]
    if not ordered_nodes_match(path["node_ids"], groups):
        return False
    assert len(path["hop_edge_ids"]) == len(reference["edge_requirements"])
    return all(any(edge_matches(edges[eid], source, target, requirement) for eid in edge_ids)
               for source, target, edge_ids, requirement in zip(
                   path["node_ids"], path["node_ids"][1:], path["hop_edge_ids"], reference["edge_requirements"]))


assert len({r["reference_id"] for r in references}) == len(references)
exact_rows = []
# Load one response at a time, and only when a complete reference is present.
for case_id in by_case:
    case_refs = [r for r in references if r["case_id"] == case_id]
    edges = None
    for reference in case_refs:
        missing = validate_reference(reference)
        ranks, graph_ids = [], []
        if not missing:
            if edges is None:
                response_files = {p["response_file"] for p in by_case[case_id]}
                assert len(response_files) == 1
                edges = json.loads((ROOT / next(iter(response_files))).read_text())["message"]["knowledge_graph"]["edges"]
            for path in sorted(by_case[case_id], key=lambda p: p["rank"]):
                if exact_path_matches(path, reference, edges):
                    ranks.append(path["rank"])
                    graph_ids.append(path["graph_id"])
        exact_rows.append({"reference_id": reference["reference_id"], "case_id": case_id,
                           "asset_id": reference.get("asset_id"),
                           "status": "missing_reference" if missing else ("found" if ranks else "not_found_in_returned_paths"),
                           "missing_fields": "; ".join(missing),
                           "returned_paths_checked": 0 if missing else len(by_case[case_id]),
                           "first_rank": min(ranks) if ranks else None,
                           "all_matching_ranks": ranks, "matching_graph_ids": graph_ids})
assert len(exact_rows) == len(references), "Unknown case in references"
exact_results = pd.DataFrame(exact_rows)
display(exact_results.groupby("status").size().rename("reference_count").reset_index())
evaluable = exact_results[exact_results.status != "missing_reference"]
if len(evaluable):
    print(f"Exact recovery among defined references: {(evaluable.status == 'found').sum()}/{len(evaluable)}")
else:
    print("Exact recovery rate: undefined (0 complete reference paths).")
with pd.option_context("display.max_rows", 40, "display.max_colwidth", None):
    display(exact_results[["reference_id", "status", "first_rank", "missing_fields"]])


,status,reference_count
0,missing_reference,32


Exact recovery rate: undefined (0 complete reference paths).


,reference_id,status,first_rank,missing_fields
0,TestCase_0/Asset_2,missing_reference,None,ordered_nodes; edge_requirements; reference_source
1,TestCase_0/Asset_3,missing_reference,None,ordered_nodes; edge_requirements; reference_source
2,TestCase_0/Asset_5,missing_reference,None,ordered_nodes; edge_requirements; reference_source
3,TestCase_0/Asset_1,missing_reference,None,ordered_nodes; edge_requirements; reference_source
4,TestCase_6/Asset_10,missing_reference,None,ordered_nodes; edge_requirements; reference_source
5,TestCase_6/Asset_8,missing_reference,None,ordered_nodes; edge_requirements; reference_source
6,TestCase_6/Asset_6,missing_reference,None,ordered_nodes; edge_requirements; reference_source
7,TestCase_6/Asset_7,missing_reference,None,ordered_nodes; edge_requirements; reference_source
8,TestCase_2/Asset_71,missing_reference,None,ordered_nodes; edge_requirements; reference_source
9,TestCase_2/Asset_70,missing_reference,None,ordered_nodes; edge_requirements; reference_source


## Save the evaluation

The original harness ranks and the new exact-path status are separate outputs. A missing reference is never encoded as a failed retrieval or a zero rank.

In [5]:
assets.to_csv(OUT / "exact_path_asset_audit.csv", index=False)
candidate_results.to_csv(OUT / "imatinib_candidate_node_sequence_check.csv", index=False)
exact_results.to_csv(OUT / "exact_path_evaluation.csv", index=False)
exact_results.to_json(OUT / "exact_path_evaluation.json", orient="records", indent=2)
print("Saved the asset audit, candidate node-sequence check, and exact-reference evaluation.")


Saved the asset audit, candidate node-sequence check, and exact-reference evaluation.
